In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv(r'MAAD_Face.csv')

C:\Users\alejo\AppData\Local\Temp\ipykernel_9556\764656773.py:4: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'MAAD_Face.csv')


In [3]:
ATTRS = ['Male', 'Young', 'Senior','Asian', 'Black', 'Smiling',
         'Blond_Hair', 'Chubby', 'Heavy_Makeup','Black_Hair', 'Big_Nose']

missing = [a for a in ATTRS if a not in df.columns]
print("Missing:", missing)

assert len(missing) == 0, "Te faltan columnas en df"

work = df[['Filename', 'Identity'] + ATTRS].copy()
for a in ATTRS:
    work[a] = (work[a] == 1).astype(np.int8)

pos_freq = work[ATTRS].mean()

weights = np.zeros(len(work))

for a in ATTRS:
    peso_positivo = 1.0 / pos_freq[a]
    peso_negativo = 1.0 / (1.0 - pos_freq[a])
    
    weights += np.where(work[a] == 1, peso_positivo, peso_negativo)

weights = weights / weights.sum()

N_IMAGES_TO_DOWNLOAD = 100000 
df_balanced = work.sample(n=N_IMAGES_TO_DOWNLOAD, weights=weights, random_state=42)

print("Nuevas proporciones de la clase positiva en la muestra balanceada:")
print(df_balanced[ATTRS].mean() * 100)

Missing: []
Nuevas proporciones de la clase positiva en la muestra balanceada:
Male            56.589
Young           39.538
Senior          11.596
Asian            7.895
Black            9.231
Smiling         23.068
Blond_Hair      13.066
Chubby          17.751
Heavy_Makeup    32.564
Black_Hair      21.470
Big_Nose        22.707
dtype: float64


In [4]:
corr_matrix = df_balanced[ATTRS].corr()
np.fill_diagonal(corr_matrix.values, 0)
pares_corr = corr_matrix.unstack().sort_values(key=abs, ascending=False).drop_duplicates()
focos_rojos = pares_corr[abs(pares_corr) > 0.25]

print("Correlaciones fuertes (Posibles problemas de Entanglement):")
print(focos_rojos)

Correlaciones fuertes (Posibles problemas de Entanglement):
Male          Heavy_Makeup   -0.793007
              Young          -0.617914
Young         Heavy_Makeup    0.603738
Heavy_Makeup  Smiling         0.565120
Male          Smiling        -0.552640
Black         Big_Nose        0.520909
Chubby        Senior          0.483294
Black_Hair    Asian           0.444965
Male          Blond_Hair     -0.440058
Chubby        Big_Nose        0.436747
Heavy_Makeup  Blond_Hair      0.418346
Male          Big_Nose        0.414720
Young         Smiling         0.387693
              Chubby         -0.372784
Big_Nose      Young          -0.367564
Chubby        Male            0.359845
Big_Nose      Heavy_Makeup   -0.355914
Blond_Hair    Smiling         0.340327
Chubby        Heavy_Makeup   -0.316180
Senior        Male            0.302846
              Young          -0.292876
Heavy_Makeup  Senior         -0.250743
dtype: float64


In [6]:
df_balanced.to_csv('df_balanced.csv')

--- 

In [10]:
import zipfile
import os
import pandas as pd

ruta_zip_origen = r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\stable-difussion\vggface2.zip"
ruta_zip_destino = r"./dataset_filtrado_drive.zip"

df_balanced = pd.read_csv("./df_balanced.csv") 
objetivos = set(df_balanced["Filename"].astype(str))  

print(f"Objetivos: {len(objetivos)}")

guardadas = 0
with zipfile.ZipFile(ruta_zip_origen, "r") as zin, \
     zipfile.ZipFile(ruta_zip_destino, "w", compression=zipfile.ZIP_DEFLATED) as zout:

    for info in zin.infolist():
        name = info.filename
        if not name.lower().endswith(".jpg"):
            continue

        norm = name.replace("\\", "/")
        tail = "/".join(norm.split("/")[-2:]) 
        idx = norm.find("/n")
        if idx != -1:
            rel = norm[idx+1:] 
        else:
            rel = norm

        if rel in objetivos:
            zout.writestr(info, zin.read(info))
            guardadas += 1
            if guardadas % 5000 == 0:
                print("Guardadas:", guardadas)

print("Total guardadas:", guardadas)

Objetivos: 100000
Guardadas: 5000
Guardadas: 10000
Guardadas: 15000
Guardadas: 20000
Guardadas: 25000
Guardadas: 30000
Guardadas: 35000
Guardadas: 40000
Guardadas: 45000
Guardadas: 50000
Guardadas: 55000
Guardadas: 60000
Guardadas: 65000
Guardadas: 70000
Guardadas: 75000
Guardadas: 80000
Guardadas: 85000
Guardadas: 90000
Total guardadas: 94769


In [11]:
import os

path = r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\stable-difussion\dataset_filtrado_drive.zip"
size_bytes = os.path.getsize(path)

size_mb = size_bytes / (1024**2)
size_gb = size_bytes / (1024**3)

print(f"Bytes: {size_bytes:,}")
print(f"MB: {size_mb:.2f}")
print(f"GB: {size_gb:.3f}")

Bytes: 1,183,062,247
MB: 1128.26
GB: 1.102
